# Create Training Subset

This notebook creates the 30% training subset used for the XCrime-LLM fine-tuning workflow.

The subset is sampled from the full NYC training split produced by the preprocessing and feature-engineering notebook.

In [ ]:
import pandas as pd
import numpy as np
from google.colab import drive

In [ ]:
 # Mount Google Drive
drive.mount('/content/drive')

In [ ]:
# Load the full training split produced by the preprocessing notebook
TRAIN_PATH = "/content/drive/MyDrive/XCrime-LLM/data/splits/master_train.csv"

df = pd.read_csv(TRAIN_PATH)

print("Training rows:", len(df))
df.head()

### Inspect Training Data

Verify the columns and data types of the full training split before constructing the 30% subset.

In [ ]:
print("== Dataset Shape ==")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

print("\n== Columns ==")
print(df.columns.tolist())

print("\n== Dtypes ==")
print(df.dtypes)

### Inspect Label Distribution

Examine the positive and negative label distribution for each crime category in the full training split.

In [ ]:
summary = (
    df.groupby("crime")["label_7d"]
    .value_counts()
    .unstack(fill_value=0)
    .rename(columns={0: "neg_0", 1: "pos_1"})
)

summary["total"] = summary.sum(axis=1)
summary["pos_rate"] = (
    summary["pos_1"] / summary["total"]
).round(3)

print(summary)

print(
    "\nOVERALL →",
    "total:", int(summary["total"].sum()),
    "| pos:", int(summary["pos_1"].sum()),
    "| neg:", int(summary["neg_0"].sum()),
)

### Prepare Training Data

Standardize the training data types, retain the features used in XCrime-LLM, and verify that each region-date anchor contains all four crime categories.

In [ ]:
# Detect the date column
DATE_COL = (
    "date"
    if "date" in df.columns
    else ("complaint_dt" if "complaint_dt" in df.columns else None)
)

assert DATE_COL is not None, "Expected a 'date' or 'complaint_dt' column."

# Standardize data types
x = df.copy()

x[DATE_COL] = pd.to_datetime(x[DATE_COL], errors="coerce")
x["region_id"] = pd.to_numeric(
    x.get("region_id"),
    errors="coerce"
).astype("Int64")

x["crime"] = (
    x.get("crime", "")
    .astype("string")
    .str.strip()
)

for col in ["last7_total", "last28_mean", "R1_influence", "base_rate"]:
    if col in x.columns:
        x[col] = (
            pd.to_numeric(x[col], errors="coerce")
            .fillna(0.0)
            .astype(float)
        )

if "base_rate" in x.columns:
    x["base_rate"] = x["base_rate"].clip(0.0, 1.0)

if "recency" in x.columns:
    x["recency"] = pd.to_numeric(x["recency"], errors="coerce")
    x["recency"] = np.where(
        x["recency"].isna(),
        9999,
        x["recency"]
    )
    x["recency"] = np.where(
        x["recency"] == 9999,
        9999,
        np.minimum(x["recency"], 365)
    ).astype(int)

for col in ["dow", "month"]:
    if col in x.columns:
        x[col] = (
            pd.to_numeric(x[col], errors="coerce")
            .fillna(0)
            .astype(int)
        )

# Map crime categories to event types
event_mapping = {
    "BURGLARY": "EVENT_TYPE_A",
    "ROBBERY": "EVENT_TYPE_B",
    "GRAND LARCENY": "EVENT_TYPE_C",
    "FELONY ASSAULT": "EVENT_TYPE_D",
}

if "event_type" not in x.columns:
    x["event_type"] = (
        x["crime"]
        .astype("string")
        .str.upper()
        .str.strip()
        .map(event_mapping)
    )

x["date"] = pd.to_datetime(x[DATE_COL], errors="coerce")

# Retain the features used in the training-subset workflow
keep = [
    "region_id",
    "date",
    "crime",
    "event_type",
    "last7_total",
    "last28_mean",
    "recency",
    "R1_influence",
    "base_rate",
    "dow",
    "month",
    "label_7d",
]

train_long = x[
    [col for col in keep if col in x.columns]
].copy()

# Verify that every anchor contains all four crime categories
anchor_counts = (
    train_long
    .groupby(["region_id", "date"])["event_type"]
    .nunique()
)

assert (
    anchor_counts == 4
).all(), "Some (region_id, date) anchors are missing event rows."

print(
    "[OK] train_long rows:",
    len(train_long),
    "| anchors:",
    anchor_counts.size,
)

### Create the 30% Training Subset

Construct a reproducible 30% subset of training anchors using stratified sampling by month and the number of positive crime labels. The sampled anchors are then rejoined to the long-format dataset so that all four crime-category rows are retained for each selected region-date anchor.

In [ ]:
EVENTS = [
    "EVENT_TYPE_A",
    "EVENT_TYPE_B",
    "EVENT_TYPE_C",
    "EVENT_TYPE_D",
]

RANDOM_SEED = 42
SAMPLE_FRACTION = 0.30

# Create one row per region-date anchor with four event labels
train_labels_wide = (
    train_long
    .pivot_table(
        index=["region_id", "date"],
        columns="event_type",
        values="label_7d",
        aggfunc="max",
    )
    .reindex(columns=EVENTS)
    .fillna(0)
    .astype(int)
    .reset_index()
)

# Stratify anchors by month and number of positive labels
anchors = train_labels_wide.copy()
anchors["month"] = pd.to_datetime(anchors["date"]).dt.month.astype(int)
anchors["sum_pos"] = anchors[EVENTS].sum(axis=1).astype(int)

anchors["strat_key"] = (
    anchors["month"].astype(str)
    + "_"
    + anchors["sum_pos"].astype(str)
)

# Sample 30% within each stratum using a fixed random seed
sampled_idx = (
    anchors
    .groupby("strat_key", group_keys=False)
    .apply(
        lambda group: group.sample(
            frac=SAMPLE_FRACTION,
            random_state=RANDOM_SEED,
        )
    )
    .index
)

anchors_30 = anchors.loc[sampled_idx].reset_index(drop=True)

# Rejoin selected anchors to the long-format training data
anchor_keys = ["region_id", "date"]

train_long_30 = train_long.merge(
    anchors_30[anchor_keys],
    on=anchor_keys,
    how="inner",
    validate="many_to_one",
)

# Verify that each selected anchor retains all four event rows
assert (
    train_long_30
    .groupby(anchor_keys)["event_type"]
    .nunique()
    .eq(4)
    .all()
)

print(
    f"[SUBSET] anchors FULL={len(anchors):,} "
    f"| 30%={len(anchors_30):,}"
)

print(
    f"[SUBSET] long rows FULL={len(train_long):,} "
    f"| 30%={len(train_long_30):,}"
)


def audit_subset(full_long, sample_long):
    full_rates = (
        full_long
        .groupby("crime")["label_7d"]
        .mean()
        .rename("FULL")
    )

    sample_rates = (
        sample_long
        .groupby("crime")["label_7d"]
        .mean()
        .rename("SAMPLE")
    )

    comparison = pd.concat(
        [full_rates, sample_rates],
        axis=1,
    )

    comparison["DIFF_pp"] = (
        comparison["SAMPLE"] - comparison["FULL"]
    ) * 100

    print("\nPer-crime positive rates (FULL vs SAMPLE)")
    print(
        comparison.to_string(
            float_format=lambda value: f"{value:.6f}"
        )
    )


audit_subset(train_long, train_long_30)

### Save the 30% Training Subset

Rejoin the sampled anchors to the original training dataset so that the original schema is preserved, then save the 30% training subset for the fine-tuning data preparation stage.

In [ ]:
# Use the original training dataset schema
DATE_COL = "date" if "date" in df.columns else "complaint_dt"

# Extract the selected region-date anchors
anchors_30_keys = (
    train_long_30[["region_id", "date"]]
    .drop_duplicates()
    .rename(columns={"date": "_anchor_date"})
)

# Create day-level join keys
df["_date_key"] = (
    pd.to_datetime(df[DATE_COL], errors="coerce")
    .dt.floor("D")
)

anchors_30_keys["_date_key"] = (
    pd.to_datetime(
        anchors_30_keys["_anchor_date"],
        errors="coerce",
    )
    .dt.floor("D")
)

# Rejoin the selected anchors to the original training schema
df_30 = (
    df.merge(
        anchors_30_keys[["region_id", "_date_key"]],
        on=["region_id", "_date_key"],
        how="inner",
        validate="many_to_one",
    )
    .drop(columns=["_date_key"])
)

# Save the 30% training subset
OUTPUT_PATH = (
    "/content/drive/MyDrive/XCrime-LLM/data/splits/"
    "master_train_30pct.csv"
)

df_30.to_csv(OUTPUT_PATH, index=False)

# Verify anchor counts
n_sampled_anchors = len(anchors_30_keys)

n_saved_anchors = (
    df_30[["region_id", DATE_COL]]
    .assign(
        _date_key=pd.to_datetime(
            df_30[DATE_COL],
            errors="coerce",
        ).dt.floor("D")
    )
    .drop_duplicates(subset=["region_id", "_date_key"])
    .shape[0]
)

assert n_sampled_anchors == n_saved_anchors

print("Saved:", OUTPUT_PATH)
print(
    "Anchors:",
    n_sampled_anchors,
    "| Rows:",
    f"{len(df_30):,}",
)

### Validate the Training Subset

Verify that the sampled dataset contains approximately 30% of the original anchors and preserves the monthly and per-crime label distributions.

In [ ]:
# Create day-level anchor keys for validation
full = df.copy()
subset = df_30.copy()

FULL_DATE = "date" if "date" in full.columns else "complaint_dt"
SUBSET_DATE = "date" if "date" in subset.columns else "complaint_dt"

full["_date_key"] = (
    pd.to_datetime(full[FULL_DATE], errors="coerce")
    .dt.floor("D")
)

subset["_date_key"] = (
    pd.to_datetime(subset[SUBSET_DATE], errors="coerce")
    .dt.floor("D")
)

# Anchor coverage
full_anchors = (
    full[["region_id", "_date_key"]]
    .dropna()
    .drop_duplicates()
)

subset_anchors = (
    subset[["region_id", "_date_key"]]
    .dropna()
    .drop_duplicates()
)

sample_percentage = (
    100 * len(subset_anchors) / len(full_anchors)
)

print("=== Anchor Coverage ===")
print(f"Full anchors:   {len(full_anchors):,}")
print(f"Subset anchors: {len(subset_anchors):,}")
print(f"Subset size:    {sample_percentage:.2f}%")

# Confirm every sampled anchor exists in the full training set
anchor_check = subset_anchors.merge(
    full_anchors,
    on=["region_id", "_date_key"],
    how="left",
    indicator=True,
)

assert (anchor_check["_merge"] == "both").all()

# Compare per-crime positive rates
print("\n=== Per-Crime Positive Rates ===")

full_rates = (
    full.groupby("crime")["label_7d"]
    .mean()
    .rename("FULL")
)

subset_rates = (
    subset.groupby("crime")["label_7d"]
    .mean()
    .rename("SUBSET")
)

rate_comparison = pd.concat(
    [full_rates, subset_rates],
    axis=1,
)

rate_comparison["DIFF_pp"] = (
    rate_comparison["SUBSET"]
    - rate_comparison["FULL"]
) * 100

print(
    rate_comparison.to_string(
        float_format=lambda value: f"{value:.4f}"
    )
)

# Compare month distribution
print("\n=== Month Distribution (%) ===")

full_month = (
    pd.to_datetime(full[FULL_DATE], errors="coerce")
    .dt.month
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .rename("FULL")
)

subset_month = (
    pd.to_datetime(subset[SUBSET_DATE], errors="coerce")
    .dt.month
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .rename("SUBSET")
)

month_comparison = pd.concat(
    [full_month, subset_month],
    axis=1,
).round(2)

print(month_comparison)

print("\nTraining-subset validation passed.")